# PS S6E6 - 031 Blend add030 OVR XGB check

Experiment: `exp_20260608_031_blend_add030_ovr_xgb_check`

Purpose:
- Add `030 one-vs-rest XGB as-is` to the current candidate pool.
- Check whether 030 adds useful low-correlation / error-complement signal to the 029 best blend and its core components.
- Keep this as a low-cost diagnostic. 030 single is weak, so the key question is whether it receives non-zero safe blend weight.

Current context:
- 029 best: `T_027_026_028 / hc_nonnegative_raw`
  - CV: `0.9700023026377228`
  - Public LB: `0.970036`
- 030 single:
  - CV: `0.9609971296999271`
  - Public LB: `0.96118`
  - role: weak XGB OVR material, only worth keeping if correlation/blend behavior is unexpectedly favorable.

Important:
- `LogReg / Ridge / ElasticNet` rows are diagnostic only. Final candidates should come from safe methods (`avg`, `hc_nonnegative_raw`, `nm_softmax_raw`) unless a fold-safe nested stacker is built.
- Do not bias-search 030.
- If 030 weight is ~0 and CV does not improve over 029, drop 030 and proceed to 032 TabM.


In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260608_031_blend_add030_ovr_xgb_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {
        "key": "015_realmlp_seed42",
        "exp_id": "exp_20260605_015_shared001_realmlp_as_is",
        "family": "RealMLP",
        "role": "stable_single_realmlp_backup",
        "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "cv": 0.9681693449222352,
        "public_lb": 0.96977,
    },
    {
        "key": "017_realmlp_bias",
        "exp_id": "exp_20260606_017_bias_search_on_015_realmlp",
        "family": "RealMLP",
        "role": "previous_best_realmlp_bias_backup",
        "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "cv": 0.9683022653955233,
        "public_lb": 0.96985,
    },
    {
        "key": "018_xgb_shared003",
        "exp_id": "exp_20260606_018_shared003_xgb_as_is",
        "family": "XGBoost",
        "role": "effective_blend_material",
        "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "cv": 0.965207986418628,
        "public_lb": 0.96578,
    },
    {
        "key": "019_blend_best",
        "exp_id": "exp_20260607_019_blend_add018_xgb_check",
        "family": "Blend",
        "role": "previous_public_confirmed_best",
        "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "cv": 0.968872315017554,
        "public_lb": 0.97000,
    },
    {
        "key": "020_blend_bias",
        "exp_id": "exp_20260607_020_bias_search_on_019_best_blend",
        "family": "Blend",
        "role": "cv_trusted_attack_reference",
        "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "cv": 0.9692240443617589,
        "public_lb": 0.96969,
    },
    {
        "key": "024_seed_ensemble_blend",
        "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check",
        "family": "Blend",
        "role": "seed_ensemble_blend_reference",
        "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "cv": 0.9694803109983217,
        "public_lb": 0.96958,
    },
    {
        "key": "026_realmlp_v5",
        "exp_id": "exp_20260608_026_realmlp_v5_as_is",
        "family": "RealMLP",
        "role": "realmlp_v5_single_model_candidate",
        "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "cv": 0.9690389777981231,
        "public_lb": 0.96979,
    },
    {
        "key": "027_blend_add026",
        "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check",
        "family": "Blend",
        "role": "previous_cv_trusted_slot",
        "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "cv": 0.9695425457477898,
        "public_lb": 0.96975,
    },
    {
        "key": "028_cat_v3",
        "exp_id": "exp_20260608_028_cat_v3_as_is",
        "family": "CatBoost",
        "role": "catboost_v3_family_aux_material",
        "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy",
        "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy",
        "cv": 0.9688146470512534,
        "public_lb": 0.96969,
    },
    {
        "key": "029_blend_add028",
        "exp_id": "exp_20260608_029_blend_add028_cat_v3_check",
        "family": "Blend",
        "role": "public_confirmed_and_cv_trusted_current_best",
        "oof": "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy",
        "cv": 0.9700023026377228,
        "public_lb": 0.970036,
    },
    {
        "key": "030_ovr_xgb",
        "exp_id": "exp_20260608_030_ovr_xgb_as_is",
        "family": "XGBoost",
        "role": "weak_xgb_ovr_family_material",
        "oof": "oof_exp_20260608_030_ovr_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260608_030_ovr_xgb_as_is_proba.npy",
        "cv": 0.9609971296999271,
        "public_lb": 0.96118,
    },
]

# 031 should answer one question first:
# Does weak 030 OVR XGB add complementary signal to 029 or to the 027/026/028 core?
BLEND_SETS = {
    # Singles / current references
    "A_029_only": ["029_blend_add028"],
    "B_030_only": ["030_ovr_xgb"],
    "C_027_only": ["027_blend_add026"],
    "D_026_only": ["026_realmlp_v5"],
    "E_028_only": ["028_cat_v3"],
    "F_019_only": ["019_blend_best"],
    "G_018_only": ["018_xgb_shared003"],

    # Direct add030 checks
    "H_029_030": ["029_blend_add028", "030_ovr_xgb"],
    "I_027_030": ["027_blend_add026", "030_ovr_xgb"],
    "J_028_030": ["028_cat_v3", "030_ovr_xgb"],
    "K_026_030": ["026_realmlp_v5", "030_ovr_xgb"],
    "L_019_030": ["019_blend_best", "030_ovr_xgb"],
    "M_018_030": ["018_xgb_shared003", "030_ovr_xgb"],

    # Current best and core components with 030
    "N_029_028_030": ["029_blend_add028", "028_cat_v3", "030_ovr_xgb"],
    "O_029_026_030": ["029_blend_add028", "026_realmlp_v5", "030_ovr_xgb"],
    "P_029_027_030": ["029_blend_add028", "027_blend_add026", "030_ovr_xgb"],
    "Q_027_026_028_030": ["027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb"],
    "R_019_027_028_030": ["019_blend_best", "027_blend_add026", "028_cat_v3", "030_ovr_xgb"],
    "S_018_028_030": ["018_xgb_shared003", "028_cat_v3", "030_ovr_xgb"],
    "T_029_018_030": ["029_blend_add028", "018_xgb_shared003", "030_ovr_xgb"],
    "U_029_027_026_028_030": ["029_blend_add028", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb"],

    # Broader diagnostic pools. Interpret weights carefully because composite blends and components coexist.
    "V_019_027_026_028_030": ["019_blend_best", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb"],
    "W_018_019_027_026_028_030": ["018_xgb_shared003", "019_blend_best", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb"],
    "Z_full_015_017_018_019_020_024_026_027_028_029_030": [
        "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003",
        "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend",
        "026_realmlp_v5", "027_blend_add026", "028_cat_v3",
        "029_blend_add028", "030_ovr_xgb",
    ],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))


EXP_ID: exp_20260608_031_blend_add030_ovr_xgb_check
OUTDIR: /kaggle/working/exp_20260608_031_blend_add030_ovr_xgb_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
n_models: 11
n_blend_sets: 24


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then scan /kaggle/input.

    This is intentionally more robust than 027 because 028 may be uploaded as a
    separate dataset from older npy files.
    """
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 017_realmlp_bias: (577347, 3) / (247435, 3)
loaded 018_xgb_shared003: (577347, 3) / (247435, 3)
loaded 019_blend_best: (577347, 3) / (247435, 3)
loaded 020_blend_bias: (577347, 3) / (247435, 3)
loaded 024_seed_ensemble_blend: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
loaded 029_blend_add028: (577347, 3) / (247435, 3)
loaded 030_ovr_xgb: (577347, 3) / (247435, 3)
class_names: ['GALAXY', 'QSO', 'STAR']
loaded keys: ['015_realmlp_seed42', '017_realmlp_bias', '018_xgb_shared003', '019_blend_best', '020_blend_bias', '024_seed_ensemble_blend', '026_realmlp_v5', '027_blend_add026', '028_cat_v3', '029_blend_add028', '030_ovr_xgb']


In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus table for 030.
focus_030_df = pairwise_df[(pairwise_df["left"] == "030_ovr_xgb") | (pairwise_df["right"] == "030_ovr_xgb")].copy()
focus_030_df = focus_030_df.sort_values("pearson_flat_proba").reset_index(drop=True)
display(focus_030_df)
focus_030_df.to_csv(OUTDIR / "pairwise_oof_correlation_focus_030.csv", index=False)

# Also keep a compact focus table for the current best 029.
focus_029_df = pairwise_df[(pairwise_df["left"] == "029_blend_add028") | (pairwise_df["right"] == "029_blend_add028")].copy()
focus_029_df = focus_029_df.sort_values("pearson_flat_proba").reset_index(drop=True)
display(focus_029_df)
focus_029_df.to_csv(OUTDIR / "pairwise_oof_correlation_focus_029.csv", index=False)


,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,public_confirmed_and_cv_trusted_current_best,0.970002,0.970036,0.970002,0.0,0.961315,0.975867,0.972825
1,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,previous_cv_trusted_slot,0.969543,0.969750,0.969543,0.0,0.961479,0.976439,0.970710
2,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,seed_ensemble_blend_reference,0.969480,0.969580,0.969480,0.0,0.961956,0.976174,0.970311
3,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,0.969224,0.969690,0.969224,0.0,0.961137,0.976200,0.970335
4,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,0.969039,0.969790,0.969039,0.0,0.959505,0.976431,0.971181
5,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,previous_public_confirmed_best,0.968872,0.970000,0.968872,0.0,0.965479,0.974441,0.966696
6,028_cat_v3,exp_20260608_028_cat_v3_as_is,CatBoost,catboost_v3_family_aux_material,0.968815,0.969690,0.968815,0.0,0.960099,0.974826,0.971520
7,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,0.968302,0.969850,0.968302,0.0,0.959889,0.975867,0.969150
8,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,0.968169,0.969770,0.968169,0.0,0.962234,0.976098,0.966177
9,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,effective_blend_material,0.965208,0.965780,0.965208,0.0,0.966870,0.972043,0.956711


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,026_realmlp_v5,030_ovr_xgb,0.987581,0.751435,0.022543,0.035388,0.028955,0.021039,0.485861,0.014348,0.007916
1,017_realmlp_bias,030_ovr_xgb,0.988127,0.889617,0.022371,0.035542,0.028955,0.021218,0.490255,0.014324,0.007737
2,028_cat_v3,030_ovr_xgb,0.988941,0.965161,0.021862,0.035277,0.028955,0.021330,0.497194,0.013947,0.007625
3,015_realmlp_seed42,030_ovr_xgb,0.989464,0.904945,0.020662,0.034388,0.028955,0.021493,0.513575,0.012895,0.007462
4,029_blend_add028,030_ovr_xgb,0.990184,0.822720,0.020717,0.034083,0.028955,0.021287,0.509853,0.012796,0.007668
5,027_blend_add026,030_ovr_xgb,0.990340,0.828804,0.020643,0.034163,0.028955,0.021363,0.511636,0.012800,0.007592
6,020_blend_bias,030_ovr_xgb,0.990453,0.910471,0.020724,0.034489,0.028955,0.021488,0.512158,0.013001,0.007467
7,017_realmlp_bias,018_xgb_shared003,0.990670,0.908650,0.018476,0.035542,0.033536,0.025439,0.582933,0.010103,0.008097
8,024_seed_ensemble_blend,030_ovr_xgb,0.990816,0.921454,0.020201,0.033962,0.028955,0.021479,0.518350,0.012483,0.007476
9,015_realmlp_seed42,018_xgb_shared003,0.991146,0.918772,0.017757,0.034388,0.033536,0.025227,0.590848,0.009161,0.008309


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,026_realmlp_v5,030_ovr_xgb,0.987581,0.751435,0.022543,0.035388,0.028955,0.021039,0.485861,0.014348,0.007916
1,017_realmlp_bias,030_ovr_xgb,0.988127,0.889617,0.022371,0.035542,0.028955,0.021218,0.490255,0.014324,0.007737
2,028_cat_v3,030_ovr_xgb,0.988941,0.965161,0.021862,0.035277,0.028955,0.021330,0.497194,0.013947,0.007625
3,015_realmlp_seed42,030_ovr_xgb,0.989464,0.904945,0.020662,0.034388,0.028955,0.021493,0.513575,0.012895,0.007462
4,029_blend_add028,030_ovr_xgb,0.990184,0.822720,0.020717,0.034083,0.028955,0.021287,0.509853,0.012796,0.007668
5,027_blend_add026,030_ovr_xgb,0.990340,0.828804,0.020643,0.034163,0.028955,0.021363,0.511636,0.012800,0.007592
6,020_blend_bias,030_ovr_xgb,0.990453,0.910471,0.020724,0.034489,0.028955,0.021488,0.512158,0.013001,0.007467
7,024_seed_ensemble_blend,030_ovr_xgb,0.990816,0.921454,0.020201,0.033962,0.028955,0.021479,0.518350,0.012483,0.007476
8,018_xgb_shared003,030_ovr_xgb,0.991339,0.969567,0.017707,0.033536,0.028955,0.022513,0.563147,0.011023,0.006442
9,019_blend_best,030_ovr_xgb,0.992669,0.920682,0.017269,0.032528,0.028955,0.022219,0.565883,0.010309,0.006736


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,029_blend_add028,030_ovr_xgb,0.990184,0.822720,0.020717,0.034083,0.028955,0.021287,0.509853,0.012796,0.007668
1,018_xgb_shared003,029_blend_add028,0.993841,0.859932,0.014596,0.033536,0.034083,0.026611,0.648927,0.006925,0.007472
2,015_realmlp_seed42,029_blend_add028,0.997290,0.870762,0.009627,0.034388,0.034083,0.029523,0.757994,0.004865,0.004561
3,017_realmlp_bias,029_blend_add028,0.997375,0.884010,0.009729,0.035542,0.034083,0.030039,0.758827,0.005503,0.004044
4,019_blend_best,029_blend_add028,0.998329,0.881267,0.007545,0.032528,0.034083,0.029604,0.799963,0.002924,0.004479
5,026_realmlp_v5,029_blend_add028,0.998564,0.976314,0.006989,0.035388,0.034083,0.031314,0.820654,0.004074,0.002770
6,020_blend_bias,029_blend_add028,0.998627,0.902219,0.007079,0.034489,0.034083,0.030808,0.815805,0.003681,0.003275
7,028_cat_v3,029_blend_add028,0.998697,0.874527,0.007069,0.035277,0.034083,0.031224,0.818739,0.004053,0.002860
8,024_seed_ensemble_blend,029_blend_add028,0.998828,0.903624,0.006568,0.033962,0.034083,0.030796,0.826746,0.003166,0.003287
9,027_blend_add026,029_blend_add028,0.999126,0.992820,0.005849,0.034163,0.034083,0.031253,0.844836,0.002910,0.002830


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def _weight_of(keys, weights, target_key):
    if weights is None or target_key not in keys:
        return None
    return float(dict(zip(keys, weights)).get(target_key, 0.0))

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "includes_030": "030_ovr_xgb" in keys,
        "weight_030": _weight_of(keys, weights, "030_ovr_xgb"),
        "includes_029": "029_blend_add028" in keys,
        "weight_029": _weight_of(keys, weights, "029_blend_add028"),
        "includes_028": "028_cat_v3" in keys,
        "weight_028": _weight_of(keys, weights, "028_cat_v3"),
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models. Diagnostic only.
    for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(80))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_030_blend_df = safe_blend_df[safe_blend_df["includes_030"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_030_blend_df.head(80))
focus_030_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_030.csv", index=False)



=== A_029_only ['029_blend_add028'] ===

=== B_030_only ['030_ovr_xgb'] ===

=== C_027_only ['027_blend_add026'] ===

=== D_026_only ['026_realmlp_v5'] ===

=== E_028_only ['028_cat_v3'] ===

=== F_019_only ['019_blend_best'] ===

=== G_018_only ['018_xgb_shared003'] ===

=== H_029_030 ['029_blend_add028', '030_ovr_xgb'] ===

=== I_027_030 ['027_blend_add026', '030_ovr_xgb'] ===

=== J_028_030 ['028_cat_v3', '030_ovr_xgb'] ===

=== K_026_030 ['026_realmlp_v5', '030_ovr_xgb'] ===

=== L_019_030 ['019_blend_best', '030_ovr_xgb'] ===

=== M_018_030 ['018_xgb_shared003', '030_ovr_xgb'] ===

=== N_029_028_030 ['029_blend_add028', '028_cat_v3', '030_ovr_xgb'] ===

=== O_029_026_030 ['029_blend_add028', '026_realmlp_v5', '030_ovr_xgb'] ===

=== P_029_027_030 ['029_blend_add028', '027_blend_add026', '030_ovr_xgb'] ===

=== Q_027_026_028_030 ['027_blend_add026', '026_realmlp_v5', '028_cat_v3', '030_ovr_xgb'] ===

=== R_019_027_028_030 ['019_blend_best', '027_blend_add026', '028_cat_v3', '030_o

,set_name,method,keys,n_models,balanced_accuracy,includes_030,weight_030,includes_029,weight_029,includes_028,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,W_018_019_027_026_028_030,hc_nonnegative_raw,"018_xgb_shared003,019_blend_best,027_blend_add...",6,0.970033,True,2.449336e-02,False,NaN,True,...,0.972571,0.970033,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q_027_026_028_030,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,030...",4,0.970020,True,1.003170e-02,False,NaN,True,...,0.972741,0.970020,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U_029_027_026_028_030,hc_nonnegative_raw,"029_blend_add028,027_blend_add026,026_realmlp_...",5,0.970016,True,1.216199e-02,True,0.377605,True,...,0.972704,0.970016,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T_029_018_030,hc_nonnegative_raw,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.970002,True,0.000000e+00,True,1.000000,False,...,0.972825,0.970002,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T_029_018_030,nm_softmax_raw,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.970002,True,7.251357e-08,True,0.999997,False,...,0.972825,NaN,NaN,0.970002,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,O_029_026_030,ridge_reg_raw_logit,"029_blend_add028,026_realmlp_v5,030_ovr_xgb",3,0.963990,True,NaN,True,NaN,False,...,0.945433,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0,NaN
196,O_029_026_030,ridgeclf_raw_logit,"029_blend_add028,026_realmlp_v5,030_ovr_xgb",3,0.963990,True,NaN,True,NaN,False,...,0.945433,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0,NaN
197,T_029_018_030,ridge_reg_raw_logit,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.963989,True,NaN,True,NaN,False,...,0.945397,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0,NaN
198,T_029_018_030,ridgeclf_raw_logit,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.963989,True,NaN,True,NaN,False,...,0.945397,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_030,weight_030,includes_029,weight_029,includes_028,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,W_018_019_027_026_028_030,hc_nonnegative_raw,"018_xgb_shared003,019_blend_best,027_blend_add...",6,0.970033,True,2.449336e-02,False,NaN,True,...,0.972571,0.970033,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q_027_026_028_030,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,030...",4,0.970020,True,1.003170e-02,False,NaN,True,...,0.972741,0.970020,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U_029_027_026_028_030,hc_nonnegative_raw,"029_blend_add028,027_blend_add026,026_realmlp_...",5,0.970016,True,1.216199e-02,True,0.377605,True,...,0.972704,0.970016,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T_029_018_030,hc_nonnegative_raw,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.970002,True,0.000000e+00,True,1.000000,False,...,0.972825,0.970002,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T_029_018_030,nm_softmax_raw,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.970002,True,7.251357e-08,True,0.999997,False,...,0.972825,NaN,NaN,0.970002,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,G_018_only,nm_softmax_raw,018_xgb_shared003,1,0.965208,False,NaN,False,NaN,False,...,0.956711,NaN,NaN,0.965208,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
68,M_018_030,avg,"018_xgb_shared003,030_ovr_xgb",2,0.965141,True,NaN,False,NaN,False,...,0.950776,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,B_030_only,hc_nonnegative_raw,030_ovr_xgb,1,0.960997,True,1.000000e+00,False,NaN,False,...,0.937418,0.960997,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,B_030_only,nm_softmax_raw,030_ovr_xgb,1,0.960997,True,1.000000e+00,False,NaN,False,...,0.937418,NaN,NaN,0.960997,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_030,weight_030,includes_029,weight_029,includes_028,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,W_018_019_027_026_028_030,hc_nonnegative_raw,"018_xgb_shared003,019_blend_best,027_blend_add...",6,0.970033,True,2.449336e-02,False,NaN,True,...,0.972571,0.970033,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q_027_026_028_030,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,030...",4,0.970020,True,1.003170e-02,False,NaN,True,...,0.972741,0.970020,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U_029_027_026_028_030,hc_nonnegative_raw,"029_blend_add028,027_blend_add026,026_realmlp_...",5,0.970016,True,1.216199e-02,True,0.377605,True,...,0.972704,0.970016,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T_029_018_030,hc_nonnegative_raw,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.970002,True,0.000000e+00,True,1.000000,False,...,0.972825,0.970002,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T_029_018_030,nm_softmax_raw,"029_blend_add028,018_xgb_shared003,030_ovr_xgb",3,0.970002,True,7.251357e-08,True,0.999997,False,...,0.972825,NaN,NaN,0.970002,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,O_029_026_030,hc_nonnegative_raw,"029_blend_add028,026_realmlp_v5,030_ovr_xgb",3,0.970002,True,0.000000e+00,True,1.000000,False,...,0.972825,0.970002,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,H_029_030,nm_softmax_raw,"029_blend_add028,030_ovr_xgb",2,0.970002,True,3.194111e-06,True,0.999997,False,...,0.972825,NaN,NaN,0.970002,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
7,H_029_030,hc_nonnegative_raw,"029_blend_add028,030_ovr_xgb",2,0.970002,True,0.000000e+00,True,1.000000,False,...,0.972825,0.970002,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,N_029_028_030,hc_nonnegative_raw,"029_blend_add028,028_cat_v3,030_ovr_xgb",3,0.970002,True,0.000000e+00,True,1.000000,True,...,0.972825,0.970002,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,V_019_027_026_028_030,hc_nonnegative_raw,"019_blend_best,027_blend_add026,026_realmlp_v5...",5,0.969995,True,6.707447e-03,False,NaN,True,...,0.972511,0.969995,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / f"oof_{EXP_ID}_best_blend_proba.npy"
best_pred_npy = OUTDIR / f"pred_{EXP_ID}_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / f"submission_{EXP_ID}_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "includes_030": "030_ovr_xgb" in best_keys,
    "weight_030": best_row.get("weight_030"),
    "includes_029": "029_blend_add028" in best_keys,
    "weight_029": best_row.get("weight_029"),
    "includes_028": "028_cat_v3" in best_keys,
    "weight_028": best_row.get("weight_028"),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "includes_030": "030_ovr_xgb" in best_keys,
    "weight_030": best_row.get("weight_030"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "W_018_019_027_026_028_030",
  "method": "hc_nonnegative_raw",
  "keys": "018_xgb_shared003,019_blend_best,027_blend_add026,026_realmlp_v5,028_cat_v3,030_ovr_xgb",
  "n_models": 6,
  "balanced_accuracy": 0.9700333620193362,
  "includes_030": true,
  "weight_030": 0.024493364050814405,
  "includes_029": false,
  "weight_029": NaN,
  "includes_028": true,
  "weight_028": 0.45360011102950976,
  "weights_json": "{\"018_xgb_shared003\": 0.0, \"019_blend_best\": 0.0, \"027_blend_add026\": 0.20898708434879643, \"026_realmlp_v5\": 0.31291944057087945, \"028_cat_v3\": 0.45360011102950976, \"030_ovr_xgb\": 0.024493364050814405}",
  "recall_GALAXY": 0.9617383702447811,
  "recall_QSO": 0.9757902734264958,
  "recall_STAR": 0.9725714423867318,
  "hc_score_internal": 0.9700333620193362,
  "hc_n_steps": 17.0,
  "nm_score_internal": NaN,
  "nm_success": NaN,
  "nm_message": NaN,
  "C": NaN,
  "risk": NaN,
  "alpha": NaN,
  "l1_ratio": NaN
}
best_oof_score: 0.97003336201

,set_name,method,balanced_accuracy,includes_030,weight_030,submission,oof_proba,pred_proba
0,W_018_019_027_026_028_030,hc_nonnegative_raw,0.970033,True,0.024493,submission_exp_20260608_031_blend_add030_ovr_x...,oof_exp_20260608_031_blend_add030_ovr_xgb_chec...,pred_exp_20260608_031_blend_add030_ovr_xgb_che...


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "029_cv": 0.9700023026377228,
    "029_public_lb": 0.970036,
    "027_cv": 0.9695425457477898,
    "027_public_lb": 0.96975,
    "026_cv": 0.9690389777981231,
    "026_public_lb": 0.96979,
    "028_cv": 0.9688146470512534,
    "028_public_lb": 0.96969,
    "019_cv": 0.968872315017554,
    "019_public_lb": 0.97000,
    "018_cv": 0.965207986418628,
    "018_public_lb": 0.96578,
    "030_cv": 0.9609971296999271,
    "030_public_lb": 0.96118,
}

# Important derived checks for judging 030.
best_030_safe = focus_030_blend_df.iloc[0].to_dict() if len(focus_030_blend_df) else None
best_safe_weight_030 = best_info.get("weight_030")

judgement = {
    "reference_scores": reference_scores,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_safe_blend_with_030": best_030_safe,
    "main_questions": {
        "does_030_add_to_029": "Check H/N/O/P/T/U safe methods and 030 weight.",
        "does_030_add_to_027_026_028_core": "Check Q/U/V/W safe methods and 030 weight.",
        "does_030_add_to_xgb_or_cat_family": "Check M/S/T/W safe methods, especially 030 with 018 and 028.",
        "should_bias_search_030_next": "No. 030 single is weak; only keep if it receives stable non-zero safe blend weight and improves CV.",
        "should_replace_submission_slot": "Only if best safe blend beats 029 CV and Public LB confirms. Otherwise keep 029 as current best.",
    },
    "automatic_view": {
        "best_safe_includes_030": bool(best_info.get("includes_030")),
        "best_safe_weight_030": best_safe_weight_030,
        "best_030_safe_cv": None if best_030_safe is None else float(best_030_safe["balanced_accuracy"]),
        "best_030_safe_method": None if best_030_safe is None else best_030_safe["method"],
        "best_030_safe_set": None if best_030_safe is None else best_030_safe["set_name"],
        "030_keep_rule": (
            "Keep only if 030 weight is non-trivial and best safe CV improves over 029. "
            "Otherwise mark 030 as optional/drop and proceed to 032 TabM."
        ),
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 030 OVR XGB as-is to 029/027/026/028/019/018 candidate pool",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_030": focus_030_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_029": focus_029_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_030": focus_030_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add030_ovr_xgb_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 030 OVR XGB",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-08",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 030 OVR XGB as-is, "
            "and decide whether this weak XGB OVR material adds complementary signal to 029 or to the 027/026/028 core."
        ),
        "success_criteria": [
            "load 015/017/018/019/020/024/026/027/028/029/030 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations and focus table for 030",
            "evaluate add030 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_030": "pairwise_oof_correlation_focus_030.csv",
        "pairwise_oof_correlation_focus_029": "pairwise_oof_correlation_focus_029.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_030": "safe_blend_diagnostics_focus_030.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": "blend_add030_ovr_xgb_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "030 is a weak OVR XGB single model and should not be promoted unless it improves safe blend CV "
            "or receives meaningful non-zero safe blend weight with 029/core components."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review individual_scores.csv",
            "Review pairwise_oof_correlation_focus_030.csv, especially 030 vs 029/028/018",
            "Review safe_blend_diagnostics_focus_030.csv",
            "If best safe blend does not improve over 029 or 030 weight is near zero, drop 030 and proceed to 032 TabM",
            "Do not bias-search 030",
            "Do not treat in-sample LogReg/Ridge/ElasticNet rows as final candidates",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add030_ovr_xgb_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_029.csv
 - pairwise_oof_correlation_focus_030.csv
 - pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_030.csv
 - saved_submission_candidates.csv
 - submission_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend.csv
